# 02. Deep-learning pipeline

Reproduces the deployed deep-learning pipeline of Section 6.3 of the report.

Architecture:

- **DeepLOB** CNN-Inception spatial encoder over the visible 59-minute LOB window
- **Bidirectional LSTM** temporal encoder
- **Autoregressive LSTM decoder with additive (Bahdanau) attention** that emits a 60-second mid-price trajectory
- **Predictive scheduler** that converts the mid-price trajectory into a 60-position schedule (directional softmax over the last K seconds, blended with last-K TWAP)

Total trainable parameters: about 700,000.

This notebook trains one model per pair on the canonical SHA-1 holdout's dev partition and evaluates on the holdout. Training is per-pair and takes roughly 30 to 60 minutes on a Colab T4 GPU per pair.

## 0. Setup

In [ ]:
import random
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from execution_edge.splits import compute_holdout_partition, per_symbol_split
from execution_edge.preprocessing import normalize_last_minute_frame
from execution_edge.models.seq2seq_attention import Seq2SeqAttention
from execution_edge.predictive_scheduler import ScheduleConfig, build_schedule_from_forecasts
from execution_edge.walk_the_book import simulate_walk_the_book
from execution_edge.bps import compute_bps
from execution_edge.data import (
    DATA_DIR, ASK_PRICE_COLS, ASK_VOL_COLS, BID_PRICE_COLS, BID_VOL_COLS,
)

SYMBOLS = ["BTCUSDT","ETHUSDT","LTCUSDT","SOLUSDT","ADAUSDT","DOGEUSDT","XRPUSDT"]

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

dev_ids, holdout_ids = compute_holdout_partition(DATA_DIR, SYMBOLS, fraction=0.20)
print(f"holdout pool: {len(holdout_ids)} ids")

## 1. Per-pair parameters

`DEV_K` is the K selected on dev (sweep K=1..60, pick the K that minimises mean TWAP-K BPS). `DEV_ALPHA` is the predictive-scheduler blend weight, also selected on dev by a small grid search.

In [ ]:
DEV_K_ASK = {"BTCUSDT": 20, "ETHUSDT": 30, "LTCUSDT": 28, "SOLUSDT": 20,
             "ADAUSDT": 17, "DOGEUSDT": 34, "XRPUSDT": 20}
DEV_K_BID = {"BTCUSDT": 14, "ETHUSDT": 31, "LTCUSDT": 10, "SOLUSDT": 10,
             "ADAUSDT": 10, "DOGEUSDT": 21, "XRPUSDT": 20}
DEV_ALPHA_ASK = {"BTCUSDT": 0.5, "ETHUSDT": 0.9, "LTCUSDT": 1.0, "SOLUSDT": 0.5,
                 "ADAUSDT": 0.5, "DOGEUSDT": 1.0, "XRPUSDT": 0.5}
DEV_ALPHA_BID = {"BTCUSDT": 1.0, "ETHUSDT": 0.5, "LTCUSDT": 1.0, "SOLUSDT": 1.0,
                 "ADAUSDT": 0.1, "DOGEUSDT": 0.1, "XRPUSDT": 1.0}

## 2. Model

The deployed pipeline is `Seq2SeqAttention` (DeepLOB CNN + BiLSTM + autoregressive attention decoder).

In [ ]:
model = Seq2SeqAttention(hidden=128, horizon=60, dropout=0.1).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {n_params:,}")

## 3. Data preprocessing

Per pair: load X_train and y_train, attach mid-price, compute per-instrument z-score statistics on the dev partition, build tensors of shape `[input_window, n_ids, n_features]` and `[60, n_ids, n_features]`.

In [ ]:
LOB_COLS = []
for i in range(1, 6):
    LOB_COLS.extend([f"ask_price_{i}", f"ask_vol_{i}", f"bid_price_{i}", f"bid_vol_{i}"])
FEATURE_COLS = LOB_COLS + ["mid_price"]
INPUT_WINDOW = 600
TARGET_WINDOW = 60


def add_mid_price(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["mid_price"] = (df["ask_price_1"] + df["bid_price_1"]) / 2.0
    return df


def build_tensor(df: pd.DataFrame, ids: list[int], seq_len: int,
                 feature_means: np.ndarray | None = None,
                 feature_stds: np.ndarray | None = None):
    """Return [seq_len, n_ids, n_features] tensor + per-id stats."""
    df = df[df["anonymized_id"].isin(ids)].sort_values(
        ["anonymized_id", "time_in_hour"]).reset_index(drop=True)
    feat = df[FEATURE_COLS].astype(np.float32).to_numpy()
    n_ids = df["anonymized_id"].nunique()
    n_features = feat.shape[1]
    # Reshape to [n_ids, seq_len, n_features] then transpose
    arr = feat.reshape(n_ids, seq_len, n_features)
    arr_t = arr.transpose(1, 0, 2)  # [seq_len, n_ids, n_features]
    if feature_means is None:
        means = arr_t.mean(axis=0, keepdims=True)  # per-id
        stds = arr_t.std(axis=0, keepdims=True)
        stds[stds == 0] = 1.0
    else:
        means, stds = feature_means, feature_stds
    arr_z = (arr_t - means) / stds
    return torch.from_numpy(arr_z), means, stds


class TensorDataset(Dataset):
    """Pairs (visible window x [-INPUT_WINDOW:], next-minute mid-price target)."""

    def __init__(self, X_tensor: torch.Tensor, Y_tensor: torch.Tensor, mid_idx: int):
        # X_tensor: [seq_len, n_ids, n_features]
        # Y_tensor: [60, n_ids, n_features]
        self.X = X_tensor[-INPUT_WINDOW:]   # [INPUT_WINDOW, n_ids, n_features]
        self.Y_mid = Y_tensor[:, :, mid_idx]  # [60, n_ids]
        self.n_ids = X_tensor.size(1)

    def __len__(self): return self.n_ids

    def __getitem__(self, idx):
        return self.X[:, idx, :], self.Y_mid[:, idx]

## 4. Training driver

Adam, lr=1e-3, weight_decay=1e-5, batch size 32, MSE + first-difference smoothness loss with lambda=0.01, 100 epochs.

In [ ]:
def smoothness_loss(pred, target, lam=0.01):
    mse = ((pred - target) ** 2).mean()
    smooth = ((pred[:, 1:] - pred[:, :-1]) ** 2).mean()
    return mse + lam * smooth


def train_one_pair(symbol: str, epochs: int = 100, batch_size: int = 32, lr: float = 1e-3):
    x = pd.read_parquet(DATA_DIR / symbol / "X_train.parquet")
    y = pd.read_parquet(DATA_DIR / symbol / "y_train.parquet")
    x = add_mid_price(x); y = add_mid_price(y)

    ids = sorted({int(i) for i in x["anonymized_id"].astype("uint64").unique()})
    dev_sym, held_sym = per_symbol_split(ids, holdout_ids)

    X_dev, means, stds = build_tensor(x, dev_sym, seq_len=3540)
    Y_dev, _, _ = build_tensor(y, dev_sym, seq_len=60, feature_means=means, feature_stds=stds)
    X_held, _, _ = build_tensor(x, held_sym, seq_len=3540, feature_means=means, feature_stds=stds)
    Y_held, _, _ = build_tensor(y, held_sym, seq_len=60, feature_means=means, feature_stds=stds)

    mid_idx = FEATURE_COLS.index("mid_price")
    train_ds = TensorDataset(X_dev, Y_dev, mid_idx)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model = Seq2SeqAttention(hidden=128, horizon=60, dropout=0.1).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    for epoch in range(epochs):
        model.train(); total = 0.0; n = 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            pred = model(x_batch, y_teacher=y_batch)
            loss = smoothness_loss(pred, y_batch, lam=0.01)
            loss.backward(); opt.step()
            total += float(loss.item()) * x_batch.size(0); n += x_batch.size(0)
        if (epoch + 1) % 10 == 0:
            print(f"{symbol} epoch {epoch+1}: train loss {total/n:.4f}")

    return model, means, stds, X_held, held_sym


# Train on BTC (run on GPU):
# model, means, stds, X_held, held_sym = train_one_pair("BTCUSDT", epochs=100)

## 5. Inference and predictive scheduler

For each holdout hour: predict the 60-second mid-price trajectory, denormalise back to raw price space using the per-instrument scaling, feed to the predictive scheduler with `(window=DEV_K[pair], alpha=DEV_ALPHA[pair])`, score with the walk-the-book simulator.

In [ ]:
def evaluate_on_holdout(model, means, stds, X_held, held_sym, symbol: str, side: str = "ask"):
    K = DEV_K_ASK[symbol] if side == "ask" else DEV_K_BID[symbol]
    alpha = DEV_ALPHA_ASK[symbol] if side == "ask" else DEV_ALPHA_BID[symbol]

    vol = float(re.search(r"([\d.]+)", (DATA_DIR / symbol / "vol_to_fill.txt").read_text()).group(1))
    cfg = ScheduleConfig(window=K, alpha=alpha, price_cap=3.0)

    # Predict mid-price on each holdout hour
    model.eval()
    X_in = X_held[-INPUT_WINDOW:].to(device).transpose(0, 1)  # [n_ids, INPUT_WINDOW, n_features]
    with torch.no_grad():
        preds = model(X_in, y_teacher=None).cpu().numpy()  # [n_ids, 60]
    mid_idx = FEATURE_COLS.index("mid_price")
    mid_means = means[0, :, mid_idx]
    mid_stds = stds[0, :, mid_idx]
    preds_denorm = preds * mid_stds[:, None] + mid_means[:, None]

    # Score
    y = pd.read_parquet(DATA_DIR / symbol / "y_train.parquet")
    y_held_norm = normalize_last_minute_frame(y[y["anonymized_id"].isin(held_sym)])
    y_by_id = {int(a): hf for a, hf in y_held_norm.groupby("anonymized_id", sort=True)
               if not hf["close"].dropna().empty}

    bps_list = []
    for i, aid in enumerate(held_sym):
        if aid not in y_by_id: continue
        hf = y_by_id[aid]
        sched = build_schedule_from_forecasts(
            mid_pred=preds_denorm[i], volume_to_fill=vol, cfg=cfg,
        )
        if side == "bid": sched = -sched
        tot, vwap = simulate_walk_the_book(
            sched,
            hf[list(ASK_PRICE_COLS)].to_numpy(float),
            hf[list(ASK_VOL_COLS)].to_numpy(float),
            hf[list(BID_PRICE_COLS)].to_numpy(float),
            hf[list(BID_VOL_COLS)].to_numpy(float),
        )
        b = compute_bps(vwap, float(hf["close"].dropna().iloc[-1]), vol, tot)
        if not np.isnan(b): bps_list.append(b)
    return float(np.mean(bps_list)), len(bps_list)


# Example end-to-end run (BTC ask) on GPU:
# mean_bps, n = evaluate_on_holdout(model, means, stds, X_held, held_sym, "BTCUSDT", "ask")
# Expected: BPS around 1.284 (Table 6 ask-side BTC).

## 6. Canonical holdout results (cached)

The numbers below are the per-pair Predictive Scheduler holdout BPS at the per-pair `DEV_ALPHA` reported in Table 6 of the report.

In [ ]:
report_table = pd.DataFrame({
    "Pair":               ["BTC",  "ETH",  "LTC",  "SOL",  "ADA",  "DOGE", "XRP"],
    "TWAP-K (baseline)":  [1.274,  3.032,  5.324,  5.607,  5.586,  4.895,  3.906],
    "Predictive (alpha*)":[1.284,  3.016,  5.173,  5.614,  5.590,  4.734,  3.901],
    "alpha* (ask)":       [0.5,    0.9,    1.0,    0.5,    0.5,    1.0,    0.5],
}).set_index("Pair")
report_table